## Making a request

In [1]:
!uv pip install anthropic python-dotenv

Checked 2 packages in 90ms


In [8]:
MODEL="claude-haiku-4-5-20251001"

from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()

In [3]:
message = client.messages.create(
    model=MODEL,
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": "What is the capital of canada? Answer in one word"
        }
    ]
)

In [5]:
import json

print(message.model_dump_json(indent=2))

{
  "id": "msg_011Cf9pgZQNBAVGVUmTT7mvv",
  "container": null,
  "content": [
    {
      "citations": null,
      "text": "Ottawa",
      "type": "text"
    }
  ],
  "model": "claude-haiku-4-5-20251001",
  "role": "assistant",
  "stop_details": null,
  "stop_reason": "end_turn",
  "stop_sequence": null,
  "type": "message",
  "usage": {
    "cache_creation": {
      "ephemeral_1h_input_tokens": 0,
      "ephemeral_5m_input_tokens": 0
    },
    "cache_creation_input_tokens": 0,
    "cache_read_input_tokens": 0,
    "inference_geo": "not_available",
    "input_tokens": 18,
    "output_tokens": 4,
    "output_tokens_details": null,
    "server_tool_use": null,
    "service_tier": "standard"
  }
}


In [8]:
print (message.content[0].text)

Ottawa


## Multi-Turn conversations

In [10]:
message = client.messages.create(
    model=MODEL,
    max_tokens=1000,
    messages=[
        {
            "role": "user",
            "content": "What is the capital of canada? Answer in one word"
        },
        {
            "role": "assistant",
            "content": "Ottawa"
        },        
         {
            "role": "user",
            "content": "What is the biggest city? Answer in one word"
        },       
    ]
)

In [11]:
print (message.content[0].text)

Toronto


In [6]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content":text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content":text}
    messages.append(assistant_message)

def chat(messages):
    message = client.messages.create(
        model = MODEL,
        max_tokens = 1000,
        messages = messages
    )
    return message.content[0].text

In [24]:
messages = []
add_user_message(messages, "What is the capital of canada? Answer in one word")
response = chat(messages)

add_assistant_message(messages,response)
add_user_message(messages, "What is the biggest city? Answer in one word")
response = chat(messages)
print (response)

Toronto


## System prompts

In [2]:
def chat(messages, system=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }
    
    if system:
        params["system"] = system
    
    message = client.messages.create(**params)
    return message.content[0].text

## Temperature

In [3]:
def chat(messages, system=None, temperature=1.0):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    
    if system:
        params["system"] = system
    
    message = client.messages.create(**params)
    return message.content[0].text

## Streaming

In [9]:
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

stream = client.messages.create(
    model=MODEL,
    max_tokens=1000,
    messages=messages,
    stream=True
)

for event in stream:
    print(event)

RawMessageStartEvent(message=Message(id='msg_011CfFbeYx39TTN5KF42uXYM', container=None, content=[], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=18, output_tokens=1, output_tokens_details=None, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='#', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' F', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text='akeDB', type='text_delta'), index=0, type='content_block_de

## Structured data

In [16]:
messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")
#text = chat(messages, stop_sequences=["```"])


response = client.messages.create(
    model=MODEL,
    max_tokens=1000,
    messages=messages,
    stop_sequences=["```"]  # Standard in Anthropic client
)

TypeError: chat() got an unexpected keyword argument 'stop_sequences'